## Scraper de cronorunner.com (clasificaciones)

cronorunner.com organiza la información en tres niveles, y hay que pasar por los tres para llegar a datos completos de cada prueba:

1. **Listado** (`ultimosResultados.php?all`): 1.290 "días de carrera" (`evento.php?ref=XXX-YYYY`) con resultados publicados. Verificado en una sesión anterior.
2. **Evento** (`evento.php?ref=XXX-YYYY`): la ficha de ESE día de carrera. Aquí está la fecha, el municipio, y el dato importante -- **un mismo día de carrera casi siempre tiene varias pruebas distintas** (la carrera absoluta de 10K + varias carreras infantiles por categoría de edad: Benjamín, Alevín, Infantil, Cadete...), cada una con su propio enlace `resultados.php?ref=XXX-YYYY-ZZZZ` y su propia distancia (tabla "Horario").
3. **Resultado** (`resultados.php?ref=XXX-YYYY-ZZZZ`): la clasificación de UNA prueba concreta. Lo mejor de esta fuente está aquí: la página monta un gráfico (Chart.js) de "Total corredores por sexo", y el número ya viene calculado en el propio HTML, en una línea como `var data = [220,58]; var labels = ['General Masculina', 'General Femenina'];` -- **no hace falta abrir ningún PDF ni contar filas**, el recuento real está directamente en el texto de la página.

Así que el catálogo real no son 1.290 filas sino muchas más -- una fila por cada prueba dentro de cada día de carrera (en los 2 eventos que hemos mirado a mano, entre 5 y 6 pruebas cada uno). `scrape_cronorunner.ipynb` produce una fila por prueba, con el recuento de sexo ya incluido (como hace el scraper de ccnorte, no hace falta un notebook de limpieza aparte para eso).

**Aviso sobre la fecha:** a diferencia de RaceResult o youevent, en esta fuente SÍ hay fecha (en la ficha del evento, icono de calendario) -- la buena noticia es que munimicio y fecha están los dos ahí, así que esta fuente va a quedar más completa que las dos anteriores.

In [1]:
import re
import json
import time
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://www.cronorunner.com"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9,ca;q=0.8",
}


def obtener_html(url: str, referer: str | None = None, timeout: int = 20) -> str:
    headers = dict(HEADERS)
    if referer:
        headers["Referer"] = referer
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()
    return resp.text


### Fase 1 -- listado de eventos

`ultimosResultados.php` agrupa los eventos por fecha: cada grupo empieza con una cabecera `<h4><i class="fa fa-calendar"></i> viernes, 04 de septiembre del 2026</h4>` seguida de un bloque `<ul class="lista-eventos">` con uno o más `<li class="li-opcion">` (cada uno con su enlace `evento.php?ref=...` y su nombre en un `<h4>`). Así que el listado ya trae la fecha de cada evento, sin necesidad de visitar su ficha -- partimos el HTML por esas cabeceras de fecha y procesamos cada trozo con la fecha que le corresponde.

In [2]:
_MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12,
}
_RE_FECHA_TEXTO = re.compile(r"[^,]+,\s*(\d{1,2})\s+de\s+(\w+)\s+del?\s+(\d{4})")


def _fecha_a_iso(texto_fecha: str) -> str | None:
    m = _RE_FECHA_TEXTO.search(texto_fecha)
    if not m:
        return None
    dia, mes_texto, anio = m.groups()
    mes = _MESES.get(mes_texto.lower())
    return f"{anio}-{mes:02d}-{int(dia):02d}" if mes else None


_RE_GRUPO_FECHA = re.compile(
    r'<i class="fa fa-calendar"></i>\s*([^<]+?)\s*</h4>(.*?)(?=<i class="fa fa-calendar"></i>|\Z)',
    re.DOTALL,
)
_RE_EVENTO_BLOQUE = re.compile(r'<li class="li-opcion">.*?</li>', re.DOTALL)
_RE_EVENTO_REF = re.compile(r"evento\.php\?ref=([\w-]+)")
_RE_EVENTO_NOMBRE = re.compile(r"<h4>([^<]+)</h4>")


def parsear_listado_eventos(html: str) -> list[dict]:
    eventos = []
    for texto_fecha, trozo in _RE_GRUPO_FECHA.findall(html):
        fecha = _fecha_a_iso(texto_fecha)
        for bloque in _RE_EVENTO_BLOQUE.findall(trozo):
            m_ref = _RE_EVENTO_REF.search(bloque)
            m_nombre = _RE_EVENTO_NOMBRE.search(bloque)
            if m_ref and m_nombre:
                eventos.append({
                    "evento_ref": m_ref.group(1),
                    "nombre_evento": m_nombre.group(1).strip(),
                    "fecha_listado": fecha,
                })
    return eventos


### Fase 2 -- ficha del evento: fecha, municipio y pruebas

La fecha viene como texto libre ("viernes, 04 de septiembre del 2026"); la pasamos a ISO con un diccionario de meses en castellano. Las pruebas se sacan del bloque `id="listado-modalidades"` (enlaces a `resultados.php`) y la distancia de cada una, de la tabla "Horario" más abajo en la misma página -- se emparejan por el texto del nombre de la prueba, que aparece igual en los dos sitios.

In [3]:
_MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12,
}
_RE_FECHA = re.compile(r"fa-calendar\"></i>\s*[^,]+,\s*(\d{1,2})\s+de\s+(\w+)\s+del?\s+(\d{4})")
_RE_MUNICIPIO = re.compile(r'fa-map-marker"></i>\s*([^<]+?)\s*</li>')
_RE_MODALIDADES_BLOQUE = re.compile(r'id="listado-modalidades"[^>]*>(.*?)</div>', re.DOTALL)
_RE_ENLACE_RESULTADO = re.compile(r'resultados\.php\?ref=([\w-]+)" class="button">([^<]+)</a>')
_RE_HORARIO_FILA = re.compile(
    r'<span class="badge badge-danger">[^<]*</span></td>\s*<td>(\d+)</td>\s*<td>([^<]+)</td>'
)


def parsear_evento(html: str) -> dict:
    m_nombre = re.search(r"<h2>([^<]+)</h2>", html)
    nombre_evento = m_nombre.group(1).strip() if m_nombre else None

    m_fecha = _RE_FECHA.search(html)
    fecha = None
    if m_fecha:
        dia, mes_texto, anio = m_fecha.groups()
        mes = _MESES.get(mes_texto.lower())
        if mes:
            fecha = f"{anio}-{mes:02d}-{int(dia):02d}"

    m_municipio = _RE_MUNICIPIO.search(html)
    municipio = m_municipio.group(1).strip() if m_municipio else None

    m_bloque = _RE_MODALIDADES_BLOQUE.search(html)
    enlaces = _RE_ENLACE_RESULTADO.findall(m_bloque.group(1)) if m_bloque else []

    distancias_por_nombre = {
        nombre.strip(): int(distancia) for distancia, nombre in _RE_HORARIO_FILA.findall(html)
    }

    pruebas = []
    for resultado_ref, nombre_prueba in enlaces:
        nombre_prueba = nombre_prueba.strip()
        pruebas.append({
            "resultado_ref": resultado_ref,
            "nombre_prueba": nombre_prueba,
            "distancia_m": distancias_por_nombre.get(nombre_prueba),
        })

    return {"nombre_evento": nombre_evento, "fecha": fecha, "municipio": municipio, "pruebas": pruebas}


### Fase 3 -- resultado de una prueba: recuento por sexo

El recuento sale de la línea `var data = [...]; var labels = [...];` que alimenta el gráfico de pastel "Total corredores por sexo". Emparejamos cada valor con su etiqueta por posición (no asumimos que "Masculina" vaya siempre primero) y miramos el texto de la etiqueta para decidir si es `finisher_h` o `finisher_d`.

In [4]:
_RE_SEXO_CHART = re.compile(r"participantes-sexo'\);\s*var data = \[([^\]]*)\];\s*var labels = \[([^\]]*)\];")


def parsear_resultado(html: str) -> dict:
    m_nombre = re.search(r"<h2>([^<]+)</h2>", html)
    m_prueba = re.search(r"<h3>([^<]+)</h3>", html)
    nombre_evento = m_nombre.group(1).strip() if m_nombre else None
    nombre_prueba = m_prueba.group(1).strip() if m_prueba else None

    finisher_h = finisher_d = None
    m_sexo = _RE_SEXO_CHART.search(html)
    if m_sexo:
        valores = [int(v.strip()) for v in m_sexo.group(1).split(",") if v.strip()]
        etiquetas = [e.strip().strip("\'\"") for e in m_sexo.group(2).split(",") if e.strip()]
        for valor, etiqueta in zip(valores, etiquetas):
            et = etiqueta.lower()
            if "masculin" in et:
                finisher_h = valor
            elif "femenin" in et:
                finisher_d = valor

    return {
        "nombre_evento": nombre_evento,
        "nombre_prueba": nombre_prueba,
        "finisher_h": finisher_h,
        "finisher_d": finisher_d,
    }


### Descarga completa con checkpoint

Tres fases, tres checkpoints (archivos JSON/CSV propios) para poder interrumpir y continuar sin repetir peticiones: el listado es una sola petición, las fichas de evento son ~1.290 (una por `evento_ref`), y los resultados son varios miles (una por cada prueba dentro de cada evento, variable según cuántas sub-pruebas tenga cada día de carrera).

**Importante -- cabecera `Referer`:** a diferencia de RaceResult y youevent, cronorunner.com bloquea (`Acceso no autorizado o incorrecto`) una petición directa a `evento.php` o `resultados.php` que no lleve un `Referer` válido -- lo hemos comprobado a mano: pedir la URL "a pelo" falla, pero navegar haciendo clic desde el listado funciona. Por eso `obtener_html()` acepta un `referer` y se lo pasamos siempre: el listado como referer de cada `evento.php`, y la propia ficha del evento como referer de cada `resultados.php`. Si aun así te sale ese error al ejecutar a escala, es la primera cosa a revisar (puede que haga falta además mantener cookies de sesión con `requests.Session()`).

In [5]:
def descargar_catalogo_cronorunner(out_dir, pausa_segundos: float = 1.0, max_eventos: int | None = None) -> pd.DataFrame:
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    csv_file = out_path / "DF_CRONORUNNER_SUCIO.csv"
    checkpoint_eventos = out_path / "cronorunner_eventos_checkpoint.json"

    # Fase 1: listado (una sola petición).
    url_listado = f"{BASE_URL}/ultimosResultados.php?all"
    html_listado = obtener_html(url_listado)
    eventos = parsear_listado_eventos(html_listado)
    print(f"Fase 1: {len(eventos)} eventos en el listado")
    if max_eventos is not None:
        eventos = eventos[:max_eventos]

    filas = []
    if csv_file.exists():
        filas = pd.read_csv(csv_file, dtype=str).to_dict("records")
    eventos_hechos = set(json.loads(checkpoint_eventos.read_text())) if checkpoint_eventos.exists() else set()

    # Fases 2 y 3: para cada evento, su ficha (fecha/municipio/pruebas) y
    # luego cada prueba individual (recuento de sexo).
    for i, evento in enumerate(eventos, 1):
        evento_ref = evento["evento_ref"]
        if evento_ref in eventos_hechos:
            continue
        url_evento = f"{BASE_URL}/evento.php?ref={evento_ref}"
        try:
            # Referer = el listado, igual que si hubiéramos llegado aquí haciendo
            # clic en "Más información" desde ultimosResultados.php (ver nota de arriba).
            html_evento = obtener_html(url_evento, referer=url_listado)
            info_evento = parsear_evento(html_evento)
        except Exception as exc:
            print(f"  [{i}/{len(eventos)}] ERROR en evento {evento_ref}: {exc}")
            continue

        # Si por lo que sea la ficha no trae fecha, nos quedamos con la del listado
        # (las dos deberían coincidir siempre, las dos vienen del mismo texto de origen).
        fecha_evento = info_evento["fecha"] or evento.get("fecha_listado")

        for prueba in info_evento["pruebas"]:
            resultado_ref = prueba["resultado_ref"]
            try:
                # Referer = la ficha del evento, igual que si hubiéramos clicado en
                # el enlace de la prueba dentro de "Resultados".
                html_resultado = obtener_html(f"{BASE_URL}/resultados.php?ref={resultado_ref}", referer=url_evento)
                info_resultado = parsear_resultado(html_resultado)
            except Exception as exc:
                print(f"  [{i}/{len(eventos)}] ERROR en resultado {resultado_ref}: {exc}")
                info_resultado = {"finisher_h": None, "finisher_d": None}
            time.sleep(pausa_segundos)

            filas.append({
                "evento_ref": evento_ref,
                "resultado_ref": resultado_ref,
                "nombre_evento": info_evento["nombre_evento"] or evento["nombre_evento"],
                "nombre_prueba": prueba["nombre_prueba"],
                "fecha": fecha_evento,
                "municipio": info_evento["municipio"],
                "distancia_m": prueba["distancia_m"],
                "finisher_h": info_resultado["finisher_h"],
                "finisher_d": info_resultado["finisher_d"],
            })

        eventos_hechos.add(evento_ref)
        time.sleep(pausa_segundos)

        if i % 20 == 0:
            pd.DataFrame(filas).drop_duplicates().to_csv(csv_file, index=False)
            checkpoint_eventos.write_text(json.dumps(sorted(eventos_hechos)))
            print(f"  [{i}/{len(eventos)}] checkpoint guardado ({len(filas)} pruebas acumuladas)")

    df = pd.DataFrame(filas).drop_duplicates()
    df.to_csv(csv_file, index=False)
    checkpoint_eventos.write_text(json.dumps(sorted(eventos_hechos)))
    n_dias_carrera = df["evento_ref"].nunique()
    print(f"Total: {df.shape[0]} filas (pruebas), {n_dias_carrera} días de carrera distintos.")
    return df


### Ejecución

**No se lanza aquí a escala completa**: son ~1.290 fichas de evento + varios miles de páginas de resultado, todas con red real a `cronorunner.com` (este entorno no tiene acceso de red real a ese dominio, solo hemos podido llegar a él vía el navegador). Los tres parsers (`parsear_listado_eventos`, `parsear_evento`, `parsear_resultado`) están probados con páginas reales de 2 eventos de organizadores distintos -- ver la celda de pruebas más abajo. En tu máquina, con `max_eventos=None`, esto recorre el catálogo completo; tiene checkpoint propio, se puede interrumpir y continuar.

In [6]:
OUTPUT_DIR = Path("../../data/raw/cronorunner")

df_cronorunner = descargar_catalogo_cronorunner(OUTPUT_DIR, pausa_segundos=1.0)
df_cronorunner.head(20)


Fase 1: 1295 eventos en el listado
  [200/1295] checkpoint guardado (3163 pruebas acumuladas)
  [360/1295] checkpoint guardado (3173 pruebas acumuladas)
  [380/1295] checkpoint guardado (3226 pruebas acumuladas)
  [400/1295] checkpoint guardado (3301 pruebas acumuladas)
  [420/1295] checkpoint guardado (3380 pruebas acumuladas)
  [440/1295] checkpoint guardado (3433 pruebas acumuladas)
  [460/1295] checkpoint guardado (3502 pruebas acumuladas)
  [477/1295] ERROR en evento 624-1675: HTTPSConnectionPool(host='clickn.run', port=443): Max retries exceeded with url: /competitions/4-carrera-contra-el-cancer-de-pulmon-22/2022 (Caused by ConnectTimeoutError(<HTTPSConnection(host='clickn.run', port=443) at 0x116708f50>, 'Connection to clickn.run timed out. (connect timeout=20)'))
Total: 3539 filas (pruebas), 1244 días de carrera distintos.


,evento_ref,resultado_ref,nombre_evento,nombre_prueba,fecha,municipio,distancia_m,finisher_h,finisher_d
0,448-2212,448-2212-3895,XXII VOLTA A PEU LORIGUILLA,Junior (2009-2010),2026-09-04,Loriguilla,0,NaN,NaN
1,448-2212,448-2212-3861,XXII VOLTA A PEU LORIGUILLA,Benjamin (2017-2018),2026-09-04,Loriguilla,300,8.0,9.0
2,448-2212,448-2212-3862,XXII VOLTA A PEU LORIGUILLA,Alevin (2015-2016),2026-09-04,Loriguilla,700,7.0,5.0
3,448-2212,448-2212-3863,XXII VOLTA A PEU LORIGUILLA,Infantil (2013-2014),2026-09-04,Loriguilla,1050,1.0,2.0
4,448-2212,448-2212-3864,XXII VOLTA A PEU LORIGUILLA,Cadete (2011-2012),2026-09-04,Loriguilla,1400,2.0,1.0
5,448-2212,448-2212-3694,XXII VOLTA A PEU LORIGUILLA,10k,2026-09-04,Loriguilla,10000,220.0,58.0
6,331-2261,331-2261-3814,XXIX VOLTA A PEU LA CANYADA,Sub 10 (Nacidos en 2017 y posteriores),2026-08-29,Paterna,1000,NaN,NaN
7,331-2261,331-2261-3815,XXIX VOLTA A PEU LA CANYADA,Sub 14 (Nacidos entre 2013 y 2016),2026-08-29,Paterna,1000,NaN,NaN
8,331-2261,331-2261-3813,XXIX VOLTA A PEU LA CANYADA,"Volta a peu La Canyada (6,4km)",2026-08-29,Paterna,6400,320.0,156.0
9,306-2175,306-2175-3868,XXXIV VOLTA A PEU NOCTURNA A MANUEL,Prebenjamin (2019-2020),2026-08-28,Manuel,200,14.0,10.0
